# Cleanup: GCS Census Upload

This notebook cleans up resources created by `setup/02_upload_census_to_gcs.ipynb`.

**Resources that will be deleted:**

**1. BigQuery Dataset:**
   - `census_uk_2021` (and all tables including `ts001_utla` and `ts001_ctry_biglake`)

**2. BigQuery Connection:**
   - `census-biglake-connection` (used for BigLake external tables)

**3. Cloud Storage:**
   - GCS Bucket: `{PROJECT_ID}-census-data` (with all census files)

**⚠️  WARNING:**
- This operation cannot be undone
- All data will be permanently deleted
- All census CSV files in GCS will be deleted

**Required permissions:**
- `roles/bigquery.admin` (to delete BigQuery dataset and connection)
- `roles/storage.admin` (to delete GCS bucket)

## Section 1: Configuration & Authentication

In [ ]:
# Configuration variables
PROJECT_ID = "johnswain-1200-20260303091357"  
REGION = "us-central1"                           

# BigQuery
DATASET_ID = "census_uk_2021"
DATASET_LOCATION = "US"
CONNECTION_ID = "census-biglake-connection"

# Cloud Storage
BUCKET_NAME = f"{PROJECT_ID}-census-data"

print("📋 Configuration:")
print(f"  Project ID: {PROJECT_ID}")
print(f"  Region: {REGION}")
print(f"  Dataset to delete: {DATASET_ID}")
print(f"  Connection to delete: {CONNECTION_ID}")
print(f"  Bucket to delete: {BUCKET_NAME}")

In [ ]:
# Verify authentication
from google.auth import default
import google.auth

try:
    credentials, project = default()
    print("🔐 Authentication Status:")
    print(f"  ✅ Authenticated successfully")
    print(f"  ✅ Default project: {project}")
    
    if project != PROJECT_ID:
        print(f"  ⚠️  Note: Using PROJECT_ID ({PROJECT_ID}) instead of default ({project})")
    
except Exception as e:
    print(f"❌ Authentication failed: {e}")
    print("Please run: gcloud auth application-default login")
    raise

In [ ]:
# Install required libraries
import sys
import subprocess

print("📦 Installing required libraries...")
packages = ["google-cloud-bigquery", "google-cloud-storage", "google-cloud-resource-manager"]

for package in packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])

print("✅ Libraries installed")

In [ ]:
# Initialize clients
from google.cloud import bigquery
from google.cloud import storage

bq_client = bigquery.Client(project=PROJECT_ID)
storage_client = storage.Client(project=PROJECT_ID)

print("✅ Clients initialized:")
print("  - BigQuery client")
print("  - Cloud Storage client")

## Section 2: Delete BigQuery Dataset

Delete the UK Census BigQuery dataset created by `upload_census_to_gcs.ipynb`.

In [ ]:
# Delete UK Census BigQuery dataset
print("🗑️  Deleting UK Census BigQuery Dataset...")
print()

dataset_full_id = f"{PROJECT_ID}.{DATASET_ID}"

print(f"⚠️  About to delete dataset: {dataset_full_id}")
print()

try:
    # Check if dataset exists
    dataset = bq_client.get_dataset(dataset_full_id)
    tables = list(bq_client.list_tables(dataset_full_id))
    table_count = len(tables)
    
    print(f"   Found dataset with {table_count} tables")
    
    # Delete the dataset with delete_contents=True to delete all tables
    bq_client.delete_dataset(
        dataset_full_id,
        delete_contents=True,  # Delete all tables in the dataset
        not_found_ok=True      # Don't error if already deleted
    )
    
    print()
    print("=" * 60)
    print("✅ UK Census BigQuery dataset deleted successfully!")
    print("=" * 60)
    print(f"   Dataset: {DATASET_ID}")
    print(f"   Tables deleted: {table_count}")
    
except Exception as e:
    error_msg = str(e)
    
    if "404" in error_msg or "not found" in error_msg.lower():
        print("⚠️  Dataset not found (already deleted or never created)")
        table_count = 0
    else:
        print(f"❌ Error deleting dataset: {error_msg}")
        raise

## Section 3: Delete BigQuery Connection

Delete the BigQuery Connection created for BigLake external tables.

**Note:** The connection can be deleted independently of the dataset.

In [ ]:
# Delete BigQuery Connection
print("🗑️  Deleting BigQuery Connection...")
print()

print(f"⚠️  About to delete connection: {CONNECTION_ID}")
print()

try:
    # Try to delete using gcloud command (most reliable for connections)
    result = subprocess.run(
        ["gcloud", "bigquery", "connections", "delete", CONNECTION_ID,
         f"--location={DATASET_LOCATION}",
         f"--project={PROJECT_ID}",
         "--quiet"],
        capture_output=True,
        text=True
    )
    
    if result.returncode == 0:
        print("=" * 60)
        print("✅ BigQuery Connection deleted successfully!")
        print("=" * 60)
        print(f"   Connection: {CONNECTION_ID}")
        print(f"   Location: {DATASET_LOCATION}")
    elif "not found" in result.stderr.lower() or "does not exist" in result.stderr.lower():
        print("⚠️  Connection not found (already deleted or never created)")
    else:
        print(f"⚠️  Error deleting connection: {result.stderr}")
        print(f"   You can manually delete it with:")
        print(f"   gcloud bigquery connections delete {CONNECTION_ID} --location={DATASET_LOCATION} --project={PROJECT_ID}")
        
except Exception as e:
    error_msg = str(e)
    
    if "not found" in error_msg.lower() or "does not exist" in error_msg.lower():
        print("⚠️  Connection not found (already deleted or never created)")
    else:
        print(f"⚠️  Error deleting connection: {error_msg}")
        print()
        print(f"   You can manually delete it with:")
        print(f"   gcloud bigquery connections delete {CONNECTION_ID} --location={DATASET_LOCATION} --project={PROJECT_ID}")

## Section 4: Delete Cloud Storage Bucket

Delete the GCS bucket and all census files.

**⚠️  WARNING:** This will permanently delete all census CSV files in the bucket.

In [ ]:
# Delete GCS bucket and all contents
print("🗑️  Deleting Cloud Storage Bucket...")
print()

print(f"⚠️  About to delete bucket: {BUCKET_NAME}")
print()

try:
    # Get bucket
    bucket = storage_client.bucket(BUCKET_NAME)
    
    if bucket.exists():
        # List all blobs in the bucket
        blobs = list(bucket.list_blobs())
        blob_count = len(blobs)
        
        print(f"   Found bucket with {blob_count} files")
        print()
        
        # Delete all blobs first
        if blob_count > 0:
            print(f"   Deleting {blob_count} files...")
            for i, blob in enumerate(blobs):
                blob.delete()
                if (i + 1) % 10 == 0 or (i + 1) == blob_count:
                    print(f"      Deleted {i + 1}/{blob_count} files", end='\r')
            print()
            print(f"   ✅ All {blob_count} files deleted")
        
        # Delete the bucket itself
        bucket.delete()
        
        print()
        print("=" * 60)
        print("✅ Cloud Storage bucket deleted successfully!")
        print("=" * 60)
        print(f"   Bucket: {BUCKET_NAME}")
        print(f"   Files deleted: {blob_count}")
    else:
        print("⚠️  Bucket not found (already deleted or never created)")
        blob_count = 0
        
except Exception as e:
    error_msg = str(e)
    
    if "404" in error_msg or "not found" in error_msg.lower():
        print("⚠️  Bucket not found (already deleted or never created)")
        blob_count = 0
    else:
        print(f"❌ Error deleting bucket: {error_msg}")
        raise

## Section 5: Cleanup Summary

In [ ]:
# Summary
print()
print("=" * 70)
print("🎉 CLEANUP COMPLETED!")
print("=" * 70)
print()
print("📊 Summary of deleted resources:")
print()
print("   1. BigQuery Dataset:")
print(f"      - {DATASET_ID} (with {table_count if 'table_count' in locals() else 0} tables)")
print()
print("   2. BigQuery Connection:")
print(f"      - {CONNECTION_ID}")
print()
print("   3. Cloud Storage:")
print(f"      - Bucket: {BUCKET_NAME}")
print(f"      - Files: {blob_count if 'blob_count' in locals() else 0}")
print()
print("✅ All resources from upload_census_to_gcs.ipynb have been cleaned up!")
print()
print("🔗 Verify in Console:")
print(f"   BigQuery: https://console.cloud.google.com/bigquery?project={PROJECT_ID}")
print(f"   BigQuery Connections: https://console.cloud.google.com/bigquery/connections?project={PROJECT_ID}")
print(f"   Cloud Storage: https://console.cloud.google.com/storage/browser?project={PROJECT_ID}")